# 07 · Foundry Agent Service → Bifrost
Foundry agents can use models **behind your own gateway** (bring-your-own-model). The deployment registers the APIM gateway as an `ApiManagement` connection (`bifrost-apim`, see `infra/modules/foundry-connection.bicep`) in the Foundry project, exposing the router alias `heimdal-auto`. We create an agent on it – every agent LLM call now flows through HEIMDAL (metrics, limits, routing).

> Uses preview APIs (`azure-ai-projects` 2.x). Adjust to the latest docs if a call signature changed. Your user needs *Azure AI User* on the Foundry account – the deployment grants it to the signed-in user.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# The connection is created by Bicep with the APIM key of the agent user (black-widow, infra/config/heimdal.json).
project = AIProjectClient(endpoint=env('HEIMDAL_FOUNDRY_PROJECT_ENDPOINT'), credential=DefaultAzureCredential())
conn = project.connections.get(env('HEIMDAL_FOUNDRY_CONNECTION'))
print(conn.name, conn.type, conn.target)

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent = project.agents.create_version(
    agent_name='heimdall-watcher',
    definition=PromptAgentDefinition(model=f"{env('HEIMDAL_FOUNDRY_CONNECTION')}/heimdal-auto",
                                     instructions='You are Heimdall, guardian of the Bifrost. Be concise and a little epic.'))
oai = project.get_openai_client()
resp = oai.responses.create(input='Who guards the Bifrost and why does that matter for an API gateway?',
                            extra_body={'agent_reference': {'name': agent.name, 'type': 'agent_reference'}})
print(resp.output_text)

Check notebook 02's KQL: the agent's calls are attributed to `black-widow` and routed by the same policy.